# Phase 5 — ThyUS2Path case-level external validation (audit-gated family J)

**Frozen conventions carry over from Phases 1–4, nothing is fitted on the target:**
- masks from the **fixed K=20 TN3K support episode** zero-shot prototype pipeline
  (Phase-1 encoder + Phase-2 fusion decoder, hard gates on episode identity);
- descriptors: frozen masked-GAP (e3+e4) regimes R0 / R1p / R2p (+ oracles R1g/R2g
  **iff** the cohort supplies ground-truth masks — pre-declared);
- heads: the **3c-retrained** linear heads (`linear_heads_retrained.json`) and the ERM
  checkpoint (sanity-gated to its locked 0.9401); **JOINT-g / JOINT-img** loaded from
  the saved Phase-4 best states (v2 protocol) — this notebook contains **no optimizer
  and no backward pass anywhere**;
- thresholds/temperature from each model's own **TN5000-val raw probabilities only**
  (raw-raw convention; JOINT values recomputed from the archived val probs and
  lineage-asserted against the Phase-4 JSON).

**The new element is the audit-first intake (§1).** Family J (4 tests, Bonferroni
$\alpha = 0.05/4 = 0.0125$, 5000 paired resamples, McNemar on J1) is *pre-declared*
but executes as **inferential only under a license**:
`INFERENCE_LICENSE = AUDIT_OK and POLARITY_VERIFIED`. If any audit gate fails the
cohort degrades to descriptive with an explicit WITHHELD marker — the 3d
POOLED_WITHHELD precedent. J1 sens@spec90 R1p − ERM (PRIMARY operating-point
replication); J2 AUROC R1p − ERM (rank); J3 sens@spec90 JOINT-g − R1p (frozen vs
joint, H2-analogue); J4 AUROC R1p − R0 (grounding-over-image).

## Attachments (Kaggle datasets to add before Run-All)
- raw: `tn3k-data`, `tn5000`, `tnui-2021` (support episode + overlap hashing)
- checkpoints: Phase-1 `best_resnet34_seed42.pt`, Phase-2 `best_fusion_decoder.pt`
- Phase-3 artifacts: `phase3_classification/` (features_\*.npz,
  `linear_heads_results.json`, `erm_resnet34_tn5000.pt`, `erm_resnet34_probs.json`,
  `run_summary.json`)
- Phase-3c artifacts: `linear_heads_retrained.json`
- Phase-4 artifacts: `joint_g_trunk.pt`, `joint_img_trunk.pt`,
  `phase4_joint_probs_tn5000.json`
- **ThyUS2Path** dataset (any folder layout; the §1 audit discovers it by name token)

In [ ]:
import os, re, json, time, hashlib, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE, MASK_THRESH, PRED_THRESH = 384, 127, 0.5
SEG_K, RING_PX = 20, 15
PROTO_SCALES, SCORE_HW = ["e2", "e3", "e4"], 48
FEAT_SCALES = ["e3", "e4"]
BATCH = 16
N_BOOT_J = 5000
ALPHA_J = 0.05 / 4
ERM_REF_AUROC, R1P_REF_AUROC, GATE_TOL_AUROC = 0.9401, 0.8902, 0.002
PROTO_REF = {"TNUI-test": 0.7079, "DDTI": 0.6782}          # locked Phase-2 zero-shot refs
PREV_LO, PREV_HI = 0.20, 0.85                              # audit prevalence plausibility band
MIN_USABLE = 300                                           # audit power floor

OUTPUT_ROOT = Path("/kaggle/working/phase5_thyus2path")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def rglob1(patterns, roots=("/kaggle/input", "/kaggle/working")):
    """First existing match (files or dirs) for any glob; deterministic sort."""
    hits = []
    for pat in patterns:
        for root in roots:
            hits += sorted(Path(root).rglob(pat))
    return [h for h in hits if h.exists()]

ART = {}
def need(key, patterns, kind="file"):
    hits = rglob1(patterns)
    ART[key] = hits[0] if hits else None
    return ART[key]

need("thy_root", ["*ThyUS2Path*", "*thyus2path*", "*ThyUS2Path*/*"], kind="dir")
if ART["thy_root"] is None and Path("/kaggle/input/datasets").exists():
    for base in sorted(Path("/kaggle/input/datasets").rglob("*")):
        if base.is_dir() and "thyus" in base.name.lower():
            ART["thy_root"] = base
            break
need("ckpt_phase1", ["best_resnet34_seed42.pt"])
need("ckpt_decoder", ["best_fusion_decoder.pt"])
need("p3_summary",  ["phase3_classification/run_summary.json", "run_summary.json"])
need("lin_heads",   ["phase3_classification/linear_heads_results.json", "linear_heads_results.json"])
need("lin_retr",    ["phase3c_external/linear_heads_retrained.json", "linear_heads_retrained.json"])
need("erm_ckpt",    ["phase3_classification/erm_resnet34_tn5000.pt", "erm_resnet34_tn5000.pt"])
need("erm_probs",   ["phase3_classification/erm_resnet34_probs.json", "erm_resnet34_probs.json"])
need("tn5k_fval",   ["phase3_classification/features_val.npz", "features_val.npz"])
need("tn5k_ftest",  ["phase3_classification/features_test.npz", "features_test.npz"])
need("joint_g",     ["phase4_joint/joint_g_trunk.pt", "joint_g_trunk.pt"])
need("joint_img",   ["phase4_joint/joint_img_trunk.pt", "joint_img_trunk.pt"])
need("joint_probs", ["phase4_joint/phase4_joint_probs_tn5000.json", "phase4_joint_probs_tn5000.json"])
need("tn3k_root",   ["tn3k-data/Thyroid Dataset/tn3k", "*Thyroid Dataset/tn3k"], kind="dir")
need("tnui_root",   ["tnui-2021/TNUI-2021-/thyroid_data", "*TNUI-2021-*/thyroid_data"], kind="dir")
need("tn5000_root", ["tn5000/TN5000_forReview", "*TN5000_forReview"], kind="dir")

REQUIRED = ["thy_root", "ckpt_phase1", "ckpt_decoder", "p3_summary", "lin_heads",
            "lin_retr", "erm_ckpt", "erm_probs", "tn5k_fval", "tn5k_ftest",
            "joint_g", "joint_img", "joint_probs", "tn3k_root", "tnui_root", "tn5000_root"]
missing = [k for k in REQUIRED if ART.get(k) is None]
assert not missing, f"missing artifacts {missing} - see the attachments markdown cell"
for k in REQUIRED:
    print(f"  {k:12s} -> {ART[k]}")
print("device:", DEVICE, "| output:", OUTPUT_ROOT)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

## 1. Cohort audit (runs before any metric cell — the license gate)

**Amended 2026-08-22 (pre-inference; nothing target-side had been computed when the
original single-table parser hard-stopped):** ThyUS2Path stores labels CASE-level in
per-batch `*_image_label.csv` files (`patient_name`, `histo_label` ∈ {0,1}); images sit
flat in per-batch `dataset/` folders named `<patient>_<view>`. A generic single-table
parser therefore cannot apply — the audit below implements the documented two-table
join instead, with the same gate strengths throughout.

Five gates, each printed with evidence:
- **A1b per-batch prefix join** — case key = image-stem prefix before the first `_`,
  matched to that batch's own label table (**batch-namespaced**: patient numbers
  collide across the two cohorts, and are different patients). Strict label-token
  purity; per-batch AND overall image coverage >= 95% hard-asserted; the corpus image
  maps (`patient_name`, `path`) are printed as cross-consistency QC;
- **A2 case assembly** — batch-namespaced case table (first image, image count,
  label). Images are legitimately multi-per-case here (mean ~10); within-case
  correlation is handled at the analysis level (case-mean scores, §6), not by
  discarding images;
- **A3 cross-cohort overlap** — exact SHA256 plus near-duplicate dHash (Hamming <= 5)
  against TN5000/TNUI-2021/TN3K; a single matching image excludes the **whole case**
  (conservative);
- **A4 prevalence plausibility** — case-level malignant fraction in [0.20, 0.85];
- **A5 polarity eyeball grid** — 6 benign + 6 malignant case thumbnails.
  `POLARITY_VERIFIED` defaults **False**; operator sets True only after eyeballing
  the grid and the dataset documentation.

`AUDIT_OK` = A1 (coverage) AND A4 (prevalence), computed after the hard-gated A2/A3
exclusions; `INFERENCE_LICENSE = AUDIT_OK and POLARITY_VERIFIED` (operator-gated).
Family J is inferential only under the license.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
print("AUDIT ROOT:", ART["thy_root"])

thy_imgs = sorted(p for p in Path(ART["thy_root"]).rglob("*")
                  if p.is_file() and p.suffix.lower() in IMG_EXTS)
assert len(thy_imgs) >= MIN_USABLE, f"too few images found ({len(thy_imgs)}) - wrong root?"
print(f"A0 images: {len(thy_imgs)}")

MAL_TOK = {"1", "1.0", "malignant", "malignant.", "m", "cancer", "carcinoma", "ptc",
           "malign", "positive", "pos", "true"}
BEN_TOK = {"0", "0.0", "benign", "benign.", "b", "normal", "negative", "neg", "false"}

def norm_tok(v):
    v = str(v).strip().lower()
    return v[:-2] if v.endswith(".0") else v

def batch_key_of(p, root):
    dirs = Path(p).relative_to(root).parts[:-1]
    for part in dirs:
        if "batch" in part.lower():
            return part
    return dirs[0] if dirs else "root"

def parse_case_label_table(lf):
    "Parse a CASE-level label table: need a pure two-class label column + an id column."
    try:
        df = pd.read_csv(lf) if lf.suffix.lower() == ".csv" else pd.read_json(lf)
    except Exception:
        return None
    if not len(df):
        return None
    lab_col, lab_map = None, {}
    for c in df.columns:
        vals = df[c].map(norm_tok)
        uniq = set(vals.unique())
        if uniq and uniq <= (MAL_TOK | BEN_TOK) and (uniq & MAL_TOK) and (uniq & BEN_TOK):
            lab_col = c
            lab_map = {v: (1 if v in MAL_TOK else 0) for v in uniq}
            break
    if lab_col is None:
        return None
    id_cands = [c for c in df.columns if c != lab_col]
    id_col = next((c for c in id_cands
                   if any(t in str(c).lower() for t in ("patient", "case", "name"))), id_cands[0])
    return {"file": str(lf), "id_col": id_col, "lab_col": lab_col, "df": df, "lab_map": lab_map}

# ---- A1b per-batch prefix join (audit amendment 2026-08-22, pre-inference) ---------
root = Path(ART["thy_root"])
label_files = sorted(p for p in root.rglob("*") if p.is_file()
                     and p.suffix.lower() in {".csv", ".json"} and "label" in p.name.lower())
assert label_files, "A1 FAIL: no *label* table found anywhere in the cohort tree"
parsed = [parse_case_label_table(lf) for lf in label_files]
parsed = [p for p in parsed if p]
assert parsed, "A1 FAIL: no parseable case-label table (need a pure two-class label column)"
print("A1 label tables:")
for p_ in parsed:
    print(f"   {Path(p_['file']).name:32s} id~{p_['id_col']} lab~{p_['lab_col']} rows={len(p_['df'])}")

img_batches = {}
for pth in thy_imgs:
    img_batches.setdefault(batch_key_of(pth, root), []).append(pth)

img_rows, coverage_report = [], []
for p_ in parsed:
    df, id_col, lab_col, lab_map = p_["df"], p_["id_col"], p_["lab_col"], p_["lab_map"]
    lf_batch = batch_key_of(p_["file"], root)
    pool = img_batches.get(lf_batch, [])
    assert pool, f"A1 FAIL: label table {Path(p_['file']).name} matches no image batch ({lf_batch})"
    case_lab = {norm_tok(r[id_col]): lab_map[norm_tok(r[lab_col])] for _, r in df.iterrows()
                if norm_tok(r[lab_col]) in lab_map}
    hit = 0
    for ip in pool:
        pref = ip.stem.split("_")[0]
        if pref in case_lab:
            hit += 1
            img_rows.append({"batch": lf_batch, "prefix": pref, "case": f"{lf_batch}:{pref}",
                             "stem": ip.stem, "image": str(ip), "label": case_lab[pref]})
    cov = hit / max(len(pool), 1)
    coverage_report.append({"file": Path(p_["file"]).name, "batch": lf_batch,
                            "pool": int(len(pool)), "labelled": int(hit),
                            "coverage": round(float(cov), 4)})
    print(f"A1 {Path(p_['file']).name}: batch={lf_batch} pool={len(pool)} "
          f"labelled={hit} coverage={cov:.3f}")
    assert cov >= 0.95, f"A1 FAIL: batch {lf_batch} label coverage {cov:.3f} < 0.95"

img_df = (pd.DataFrame(img_rows).drop_duplicates(subset=["image"])
          .sort_values("image").reset_index(drop=True))
hit_rate = len(img_df) / len(thy_imgs)
n_cases = int(img_df.case.nunique())
print(f"A1 overall: {len(img_df)}/{len(thy_imgs)} images labelled ({hit_rate:.3f}); "
      f"{n_cases} cases (batch-namespaced)")
A1_OK = bool(hit_rate >= 0.95)
assert len(img_df) >= MIN_USABLE and n_cases >= MIN_USABLE, "A1 FAIL: too few labelled images/cases"

# ---- cross-consistency QC vs the corpus's own image maps (printed only) ------------
stem2pref = {r["stem"]: r["prefix"] for _, r in img_df.iterrows()}
agree = tot = 0
for mf in sorted(p for p in root.rglob("*.csv") if "label" not in p.name.lower()):
    try:
        mdf = pd.read_csv(mf)
    except Exception:
        continue
    cols = [str(c).lower() for c in mdf.columns]
    if "path" not in cols or "patient_name" not in cols:
        continue
    pcol, icol = mdf.columns[cols.index("path")], mdf.columns[cols.index("patient_name")]
    for _, r in mdf.iterrows():
        st = Path(str(r[pcol])).stem
        if st in stem2pref:
            tot += 1
            agree += int(stem2pref[st] == norm_tok(r[icol]))
if tot:
    print(f"A1 cross-consistency vs corpus image maps: {agree}/{tot} matched stems agree "
          f"on case prefix ({agree / tot:.3f})")
else:
    print("A1 cross-consistency: no corpus image-map stems matched (prefix join stands alone)")

In [ ]:
# ---- A2 case assembly (batch-namespaced; images legitimately multi-per-case) ----
case_df = (img_df.groupby("case", as_index=False)
                 .agg(first_image=("image", "min"), label=("label", "first"),
                      batch=("batch", "first"), prefix=("prefix", "first"),
                      n_images=("image", "size"))
                 .sort_values("case").reset_index(drop=True))
A2_AFTER = len(case_df)
per_batch = case_df.groupby("batch").size().to_dict()
assert A2_AFTER >= MIN_USABLE, f"A2 FAIL: {A2_AFTER} cases < {MIN_USABLE}"
ipc = case_df.n_images
print(f"A2 cases: {A2_AFTER} (per batch {per_batch}) from {len(img_df)} labelled images")
print(f"A2 images/case: min {int(ipc.min())} | mean {ipc.mean():.1f} | max {int(ipc.max())}")
prev = case_df.label.mean()
print(f"A2 cohort (case level): n={A2_AFTER} | malignant={int(case_df.label.sum())} ({prev:.3f})")
case_df.to_csv(OUTPUT_ROOT / "phase5_manifest_unique_cases.csv", index=False)

In [ ]:
# ---- A3 cross-cohort overlap: exact SHA256 + near-duplicate dHash ----------------
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def dhash64(path, size=(9, 8)):
    img = cv2.resize(np.asarray(Image.open(path).convert("L")), size,
                     interpolation=cv2.INTER_AREA)
    bits = (img[:, 1:] > img[:, :-1]).astype(np.uint8).ravel()
    return np.packbits(bits).view(np.uint64)[0]

def popcount64(a):
    return np.unpackbits(a.view(np.uint8), axis=None).reshape(a.size, 8, 8).sum(axis=2).sum(axis=1)

def collect_hashes(root, sub=()):
    root = Path(root); files = []
    for s in sub or ("images", "JPEGImages", "trainval-image", ""):
        d = root / s
        if d.exists():
            files += [p for p in d.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not files:
        files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    return files

src_files = {
    "TN5000": collect_hashes(ART["tn5000_root"]),
    "TNUI":   collect_hashes(ART["tnui_root"]),
    "TN3K":   collect_hashes(ART["tn3k_root"]),
}
src_sha, src_dh = {}, {}
for name, files in src_files.items():
    print(f"A3 hashing {name}: {len(files)} images ...", flush=True)
    for p in files:
        src_sha[p] = sha256_file(p)
        src_dh[p] = dhash64(p)
sha_index = {}
for p, h in src_sha.items():
    sha_index.setdefault(h, []).append(p)
src_dh_arr = np.array(list(src_dh.items()), dtype=object)

# conservative CASE-level exclusion: one matching image excludes the whole case
tgt = np.array([v for _, v in src_dh_arr], dtype=np.uint64)
drop_cases, near_rep, exact_rep = set(), [], []
for _, r in img_df.iterrows():
    p = Path(r["image"]); h = sha256_file(p)
    if h in sha_index:
        exact_rep.append((p.name, r["case"], [x.name for x in sha_index[h][:2]]))
        drop_cases.add(r["case"]); continue
    dh = dhash64(p)
    hd = popcount64(np.bitwise_xor(tgt, dh))
    j = int(np.argmin(hd))
    if int(hd[j]) <= 5:
        near_rep.append((p.name, r["case"], src_dh_arr[j][0].name, int(hd[j])))
        drop_cases.add(r["case"])
print(f"A3 exact-overlap: {len(exact_rep)} images | near-dup (HD<=5): {len(near_rep)} images")
for t in (exact_rep + near_rep)[:6]:
    print("   e.g.", t)
manifest5 = img_df[~img_df.case.isin(drop_cases)].reset_index(drop=True)   # per-IMAGE rows
case_df5 = case_df[~case_df.case.isin(drop_cases)].reset_index(drop=True)  # per-CASE rows
A3_EXCLUDED = len(case_df) - len(case_df5)
assert len(case_df5) >= MIN_USABLE, f"A3 FAIL: only {len(case_df5)} cases after overlap exclusion"
prev5 = case_df5.label.mean()
A4_OK = PREV_LO <= prev5 <= PREV_HI
print(f"A3 excluded {A3_EXCLUDED} cases -> cohort n_cases={len(case_df5)} "
      f"(images {len(manifest5)}; malignant cases {int(case_df5.label.sum())}, {prev5:.3f})")
print(f"A4 prevalence gate [{PREV_LO}, {PREV_HI}]: {'OK' if A4_OK else 'FAIL'}")

In [ ]:
# ---- A5 polarity grid + license computation ---------------------------------------
def readable_thumb(p):
    return np.asarray(Image.open(p).convert("L"))

rng = np.random.RandomState(SEED)
bimgs = case_df5[case_df5.label == 0].first_image.tolist()
mimgs = case_df5[case_df5.label == 1].first_image.tolist()
sel = [("benign(label=0)", bimgs), ("malignant(label=1)", mimgs)]
fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
for row, (lab_name, pool) in enumerate(sel):
    picks = rng.choice(len(pool), size=min(6, len(pool)), replace=False)
    for col in range(6):
        ax = axes[row, col]
        if col < len(picks):
            p = pool[picks[col]]
            ax.imshow(readable_thumb(p), cmap="gray")
            ax.set_title(f"{lab_name}\n{Path(p).stem[:18]}", fontsize=7)
        ax.axis("off")
fig.suptitle("ThyUS2Path polarity grid (operator: verify benign row shows typical benign US, then set POLARITY_VERIFIED=True)")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "phase5_polarity_grid.png", dpi=110)
plt.show()

POLARITY_VERIFIED = False     # <- operator sets True ONLY after eyeballing the grid + dataset docs
AUDIT_OK = bool(A1_OK and A4_OK)
INFERENCE_LICENSE = bool(AUDIT_OK and POLARITY_VERIFIED)
audit_report = {"root": str(ART["thy_root"]),
                "label_files": {Path(p_["file"]).name: {"id_col": p_["id_col"],
                                                        "lab_col": p_["lab_col"],
                                                        "lab_map": p_["lab_map"]} for p_ in parsed},
                "batch_coverage": coverage_report,
                "A1_parse_rate": float(hit_rate), "A2_images_in": int(len(img_df)),
                "A2_unique_cases": int(A2_AFTER), "A3_excluded_cases": int(A3_EXCLUDED),
                "A4_prevalence": float(prev5), "A1_OK": bool(A1_OK), "A4_OK": bool(A4_OK),
                "POLARITY_VERIFIED": bool(POLARITY_VERIFIED), "AUDIT_OK": AUDIT_OK,
                "INFERENCE_LICENSE": INFERENCE_LICENSE, "n_final": int(len(case_df5)),
                "n_images_final": int(len(manifest5)),
                "unit": "case (score = mean of per-image calibrated probabilities)",
                "amendment": ("2026-08-22 pre-inference: A1b two-table per-batch prefix join; "
                              "batch-namespaced case ids; case-mean primary unit; "
                              "conservative whole-case overlap exclusion")}
json.dump(audit_report, open(OUTPUT_ROOT / "phase5_audit_report.json", "w"), indent=2)
print("AUDIT:", json.dumps({k: audit_report[k] for k in
      ["A1_parse_rate", "A2_unique_cases", "A3_excluded_cases", "A4_prevalence",
       "A1_OK", "A4_OK", "POLARITY_VERIFIED", "AUDIT_OK", "INFERENCE_LICENSE",
       "n_final", "n_images_final"]}, indent=1))
assert A1_OK, "license cannot be granted: A1 failed (parse rate < 95%)"

## 2. Loaders + fixed support episode + frozen segmentation machinery
Identical to Phases 3c/3e/4 (byte-for-byte conventions). The TN3K support pool follows
the canonical Phase-3c chain (sha256-dedup of trainval pairs; seed-42 15%% val
carve-out); the seed-52 support episode is then re-derived and hard-checked against
the Phase-3 archived ID list.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci

def letterbox_mask(msk_raw, size=IMG_SIZE):
    h, w = msk_raw.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    rm = cv2.resize(msk_raw, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    cm = np.zeros((size, size), np.uint8); cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return cm

def make_ring(mask, r=RING_PX):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k)
    ring = np.clip(dil.astype(np.int16) - mask.astype(np.int16), 0, 1).astype(np.uint8)
    if ring.sum() == 0:
        ring = (1 - mask.astype(np.uint8))
    return ring

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

# -------- TN3K train pool and THE SAME support episode as Phases 3/3c/3e ----------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files_generic(image_dir)}
    msks = {p.stem: p for p in image_files_generic(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"id": k, "image": str(imgs[k]), "mask": str(msks[k]),
                          "dataset": dataset, "split": split} for k in common])

def image_files_generic(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

tn3k = pair_images_masks(ART["tn3k_root"]/"trainval-image", ART["tn3k_root"]/"trainval-mask",
                         "TN3K", "trainval")
# canonical Phase-3c chain: sha256-dedup, then seed-42 15% val carve-out BEFORE support pick
assert len(tn3k) > 2700, f"TN3K trainval raw pool drift: {len(tn3k)} (canonical 2879)"
tn3k["sha256"] = tn3k["image"].apply(sha256_file)
tn3k = tn3k.sort_values("id", kind="stable").drop_duplicates("sha256", keep="first")
_rs = np.random.RandomState(SEED)
_perm = _rs.permutation(len(tn3k))
_val_ids = set(tn3k.iloc[_perm[: int(round(0.15 * len(tn3k)))]].id)
tn3k["split"] = np.where(tn3k.id.isin(_val_ids), "val", "train")
tn3k_train = tn3k[tn3k.split == "train"].reset_index(drop=True)
assert len(tn3k_train) > 2000, f"TN3K support pool drift after dedup/split: {len(tn3k_train)}"
print(f"TN3K pool (canonical 3c chain): {len(tn3k)} dedup pairs -> train {len(tn3k_train)}")

def tn3k_support_rows(kmax=SEG_K, seed=SEED + 10):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(tn3k_train))[:kmax]
    return tn3k_train.iloc[idx].reset_index(drop=True)

SUP_ROWS = tn3k_support_rows()
_arch = json.loads(Path(ART["p3_summary"]).read_text())["config"]["support_episode_ids"]
assert list(SUP_ROWS.id) == _arch, "support episode mismatch vs Phase 3 archive - seed drift!"
print(f"support episode reproduced exactly ({SEG_K} ids match Phase 3 archive); pool n={len(tn3k_train)}")

def load_tn3k_item(row):
    img = letterbox_image(read_gray(row["image"]).astype(np.float32))
    msk = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    return img, msk

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class JointUNet(nn.Module):
    """Phase-1 ResNet34-UNet trunk (state-dict compatible), used FROZEN here."""
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e0": e0, "e1": e1, "e2": e2, "e3": e3, "e4": e4}
    def decode(self, f):
        d = self.up1(self.up2(self.up3(self.up4(f["e4"], f["e3"]), f["e2"]), f["e1"]), f["e0"])
        return self.head(d)
    def forward(self, x):
        f = self.features(x)
        return self.decode(f), f

def load_ckpt(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def build_frozen_trunk():
    trunk = JointUNet().to(DEVICE)
    _ck = load_ckpt(ART["ckpt_phase1"])
    trunk.load_state_dict(_ck["model"], strict=True)
    trunk.eval()
    for p in trunk.parameters():
        p.requires_grad_(False)
    print(f"  frozen Phase-1 trunk (which={_ck.get('which')}, val Dice={_ck.get('val_dice'):.4f})")
    return trunk

def encode(trunk, x):
    return trunk.features(x)

def masked_proto(feats, masks):
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    return -(feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)

class FusionDecoder(nn.Module):
    def __init__(self, cin=2 * len(PROTO_SCALES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits_384(que_feats, pf_pb_per_scale, decoder):
    return F.interpolate(decoder(score_maps(que_feats, pf_pb_per_scale)),
                         size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

trunk_frozen = build_frozen_trunk()
decoder = FusionDecoder().to(DEVICE)
_dk = load_ckpt(ART["ckpt_decoder"])
decoder.load_state_dict(_dk["decoder"], strict=True)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

_sup = [load_tn3k_item(r) for _, r in SUP_ROWS.iterrows()]
sup_x = torch.stack([to_tensor(i) for i, _ in _sup]).to(DEVICE)
sup_m = torch.stack([torch.from_numpy(m[None]) for _, m in _sup]).float().to(DEVICE)
with torch.no_grad():
    SF = encode(trunk_frozen, sup_x)
PF_PB = {name: masked_proto(SF[name], sup_m) for name in PROTO_SCALES}
print("decoder loaded | fixed support prototypes ready (3 scales)")

## 3. ThyUS2Path feature bank — zero-shot masks + frozen descriptors + seg QC
Locked conventions: hard binarisation at 0.5, <4 px empty-mask fallback (flagged),
e3+e4 masked-GAP descriptors, ring at 15 px. Oracle reps (R1g/R2g) are included
**iff** cohort masks exist (pre-declared); otherwise the bank records that gate.

In [ ]:
def masked_gap(feat, msk01):
    """Hard per-image masked GAP (locked Phase-3 convention; nearest downsample)."""
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)            # (B,1) per-image masked area
    return (feat * m).sum(dim=(2, 3)) / den           # (B, C) per-image descriptor

def dice_np(a, b, eps=1e-7):
    a = a.astype(np.uint8); b = (np.asarray(b) > 0).astype(np.uint8)
    return float((2 * (a & b).sum() + eps) / (a.sum() + b.sum() + eps))

HAS_GT_MASKS = False   # set True ONLY if the audit discovers per-image GT masks
_mdir = [p for p in Path(ART["thy_root"]).rglob("*") if p.is_dir()
         and ("mask" in p.name.lower() or "annot" in p.name.lower() or "seg" in p.name.lower())]
_img2mask = {}
if _mdir:
    cand_masks = {}
    for d in _mdir:
        for p in image_files_generic(d):
            cand_masks[p.stem] = p
    _img2mask = {p.stem: cand_masks[p.stem] for p in thy_imgs if p.stem in cand_masks}
if len(_img2mask) >= 0.95 * len(manifest5):
    HAS_GT_MASKS = True
REPS = ["R0_image", "R1p_lesion", "R2p_lesion_ring"] + (["R1g_gtmask", "R2g_gtmask_ring"] if HAS_GT_MASKS else [])
print(f"cohort GT masks available for {len(_img2mask)} stems -> HAS_GT_MASKS={HAS_GT_MASKS}; "
      f"oracle reps: {'included' if HAS_GT_MASKS else 'excluded (pre-declared)'}")

BANK_PATH = OUTPUT_ROOT / "phase5_bank_thyus2path.npz"
def build_thy_bank(df):
    if BANK_PATH.exists():
        z = np.load(BANK_PATH, allow_pickle=True)
        return {k: z[k] for k in z.files}
    store = {r: [] for r in REPS}
    conf, area, empty, dices, gtm_list = [], [], [], [], []
    ids, labels = [], []
    t0 = time.time()
    for i0 in range(0, len(df), BATCH):
        chunk = df.iloc[i0:i0 + BATCH]
        imgs, gts = [], []
        for _, r in chunk.iterrows():
            ci = letterbox_image(read_gray(r["image"]).astype(np.float32))
            imgs.append(ci)
            if HAS_GT_MASKS:
                gts.append(letterbox_mask((read_gray(_img2mask[r["stem"]]) >= MASK_THRESH).astype(np.uint8)))
        x = torch.stack([to_tensor(i) for i in imgs]).to(DEVICE)
        with torch.no_grad():
            feats = encode(trunk_frozen, x)
            prob = torch.sigmoid(predict_logits_384(feats, PF_PB, decoder))
        pm = (prob >= PRED_THRESH).float()
        pm_raw = pm.clone()
        emp = 0
        for bi in range(len(chunk)):
            if pm[bi].sum() < 4:
                pm[bi] = 1.0; emp += 1
        empty.append(emp)
        ring_p = torch.stack([torch.from_numpy(make_ring(m[0].cpu().numpy().astype(np.uint8))[None])
                              for m in pm]).float().to(DEVICE)
        combo = {"R0_image": "image", "R1p_lesion": pm, "R2p_lesion_ring": (pm, ring_p)}
        if HAS_GT_MASKS:
            gtm = torch.from_numpy(np.stack(gts)[:, None]).float().to(DEVICE)
            ring_g = torch.stack([torch.from_numpy(make_ring(gts[bi])[None])
                                  for bi in range(len(chunk))]).float().to(DEVICE)
            combo.update({"R1g_gtmask": gtm, "R2g_gtmask_ring": (gtm, ring_g)})
        desc = {}
        for rep, key in combo.items():
            if key == "image":
                desc[rep] = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1).float().cpu()
            elif isinstance(key, tuple):
                a = torch.cat([masked_gap(feats[nm], key[0]) for nm in FEAT_SCALES], 1)
                b = torch.cat([masked_gap(feats[nm], key[1]) for nm in FEAT_SCALES], 1)
                desc[rep] = torch.cat([a, b], 1).float().cpu()
            else:
                desc[rep] = torch.cat([masked_gap(feats[nm], key) for nm in FEAT_SCALES], 1).float().cpu()
        for r_ in REPS:
            store[r_].append(desc[r_].numpy())
        for bi in range(len(chunk)):
            conf.append(float(prob[bi].mean())); area.append(float(pm_raw[bi].mean()))
            if HAS_GT_MASKS:
                dices.append(dice_np(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), gts[bi]))
        ids.extend(chunk.stem.tolist()); labels.extend(chunk.label.astype(int).tolist())
        done = min(i0 + BATCH, len(df))
        if done % (BATCH * 25) < BATCH or done == len(df):
            print(f"  thy: {done}/{len(df)} images ({(time.time()-t0)/60:.1f} min)")
    out = {"ids": np.array(ids), "labels": np.array(labels, np.int64),
           "conf": np.array(conf, np.float32), "area": np.array(area, np.float32),
           "empty": np.array(empty, np.int64),
           "dice": np.array(dices if HAS_GT_MASKS else [np.nan] * len(ids), np.float32)}
    for r_ in REPS:
        out[r_] = np.concatenate(store[r_]).astype(np.float32)
    np.savez_compressed(BANK_PATH, **out)
    qc_seg = f"mean seg Dice {np.mean(dices):.4f}" if HAS_GT_MASKS else "seg Dice n/a (no cohort masks)"
    print(f"[done] thyus2path n={len(ids)} | {qc_seg} | empty-mask {sum(empty)} "
          f"({100*sum(empty)/len(ids):.1f}%) | mean area {np.mean(area):.3f} | {(time.time()-t0)/60:.1f} min")
    return out

THY_BANK = build_thy_bank(manifest5)
SEG_QC = {"dice_mean": float(np.nanmean(THY_BANK["dice"])),
          "empty_rate": float(np.mean(THY_BANK["empty"])),
          "area_mean": float(np.mean(THY_BANK["area"])),
          "conf_mean": float(np.mean(THY_BANK["conf"])),
          "has_gt_masks": bool(HAS_GT_MASKS)}

## 4. JOINT arms (saved Phase-4 states) on ThyUS2Path + seg QC
Both trunks load their saved v2 best states **strictly**; the JOINT-g mask comes from
its own decoder (hard locked eval convention: 0.5 binarisation, <4 px fallback).
Threshold/temperature lineage for the joint arms is asserted against the archived
Phase-4 JSON (\(|\Delta| \leq 10^{-6}\)).

In [ ]:
def build_head_768():
    return nn.Linear(768, 1).to(DEVICE)

def load_joint_arm(tag):
    st = load_ckpt(ART[tag])
    trk = JointUNet().to(DEVICE); hd = build_head_768()
    trk.load_state_dict(st["trunk"], strict=True); hd.load_state_dict(st["head"], strict=True)
    trk.eval(); hd.eval()
    for p in list(trk.parameters()) + list(hd.parameters()):
        p.requires_grad_(False)
    return trk, hd, st

ARMG_TRK, ARMG_HEAD, ARMG_STATE = load_joint_arm("joint_g")
ARMI_TRK, ARMI_HEAD, ARMI_STATE = load_joint_arm("joint_img")
for nm, st in [("JOINT-g", ARMG_STATE), ("JOINT-img", ARMI_STATE)]:
    assert st.get("pool_mode") in ("lesion", "image"), f"corrupt joint state for {nm}"
    print(f"  {nm}: saved epoch {st['epoch']} | val AUROC {st['val_auroc']:.4f} | "
          f"tn3k_val_dice {st.get('tn3k_val_dice', float('nan')):.4f} | protocol v2 "
          f"seg_feasible_any={st.get('seg_feasible_any')}")
assert ARMG_STATE["pool_mode"] == "lesion" and ARMI_STATE["pool_mode"] == "image"

def joint_cls_probs(trk, hd, pool_mode, df, bs=BATCH):
    """Hard-eval probs + seg QC for a joint arm (locked conventions)."""
    trk.eval(); hd.eval()
    ps, dices, empty, area = [], [], [], []
    for i0 in range(0, len(df), bs):
        chunk = df.iloc[i0:i0 + bs]
        xs, gts = [], []
        for _, r in chunk.iterrows():
            xs.append(to_tensor(letterbox_image(read_gray(r["image"]).astype(np.float32))))
            if HAS_GT_MASKS:
                gts.append(letterbox_mask((read_gray(_img2mask[r["stem"]]) >= MASK_THRESH).astype(np.uint8)))
        x = torch.stack(xs).to(DEVICE)
        with torch.no_grad():
            logits_seg, feats = trk(x)
            prob_seg = torch.sigmoid(logits_seg.float())
            if pool_mode == "lesion":
                pm = (prob_seg >= PRED_THRESH).float()
                emp = 0
                for bi in range(len(x)):
                    area.append(float(pm[bi].mean()))
                    if pm[bi].sum() < 4:
                        pm[bi] = 1.0; emp += 1
                desc = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
                empty.append(emp)
            else:
                desc = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
            ps.append(torch.sigmoid(hd(desc.float()).squeeze(1)).float().cpu())
            if HAS_GT_MASKS and pool_mode == "lesion":
                for bi in range(len(x)):
                    dices.append(dice_np(pm[bi, 0].cpu().numpy().astype(np.uint8), gts[bi]))
    qc = {"dice_mean": float(np.mean(dices)) if dices else float("nan"),
          "empty_rate": float(np.mean(empty)) if empty else 0.0,
          "area_mean": float(np.mean(area)) if area else float("nan")}
    return torch.cat(ps).numpy().ravel(), qc

P_JOINTG_EXT, JG_QC = joint_cls_probs(ARMG_TRK, ARMG_HEAD, "lesion", manifest5)
P_JOINTI_EXT, JI_QC = joint_cls_probs(ARMI_TRK, ARMI_HEAD, "image", manifest5)
print(f"JOINT-g thyus2path QC: dice {JG_QC['dice_mean']:.4f} (n/a if no masks) | "
      f"empty {100*JG_QC['empty_rate']:.1f}% | area {JG_QC['area_mean']:.3f}")

## 5. Frozen-row integrity + source-side calibration lineage (gates)
- **ERM sanity** — archived checkpoint must re-hit TN5000-test AUROC \(0.9401 \pm 0.002\);
- **R1p integrity** — 3c-retrained head on locked TN5000-test descriptors must reproduce
  \(0.8902 \pm 0.002\);
- thresholds and temperatures come from **each model's own TN5000-val raw
  probabilities** (`spec90_threshold`, `fit_temperature`); JOINT values must match the
  archived Phase-4 JSON within \(10^{-6}\).
If any gate fails the notebook stops before external metrics.

In [ ]:
ERMJ = json.loads(Path(ART["erm_probs"]).read_text())
P3S  = json.loads(Path(ART["p3_summary"]).read_text())
LINR = json.loads(Path(ART["lin_retr"]).read_text())
JP   = json.loads(Path(ART["joint_probs"]).read_text())

def labels_np(split):
    z = np.load(ART["tn5k_ftest"], allow_pickle=True) if split == "test" else np.load(ART["tn5k_fval"], allow_pickle=True)
    return z["labels"].astype(int)

y_srct = labels_np("test").astype(int)
erm_ref = roc_auc_score(y_srct, np.array(ERMJ["probs"]["test"]).ravel())
assert abs(erm_ref - ERM_REF_AUROC) <= GATE_TOL_AUROC, f"ERM sanity failed: {erm_ref:.4f}"
print(f"ERM TN5000 test AUROC sanity: {erm_ref:.4f} (ref {ERM_REF_AUROC}) OK")

r1p_ref = roc_auc_score(y_srct, np.array(LINR["R1p_lesion"]["probs"]["test"]).ravel())
assert abs(r1p_ref - R1P_REF_AUROC) <= GATE_TOL_AUROC, f"R1p integrity failed: {r1p_ref:.4f}"
print(f"R1p TN5000 test AUROC integrity: {r1p_ref:.4f} (ref {R1P_REF_AUROC}) OK")

REPS_SRC = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
MAP_EXT = {"R0_image": "R0_image", "R1p_lesion": "R1p_lesion",
           "R2p_lesion_ring": "R2p_lesion_ring", "R1g_gtbox": "R1g_gtmask",
           "R2g_gtbox_ring": "R2g_gtmask_ring"}
ACTIVE_SRC_REPS = [r for r in REPS_SRC if (HAS_GT_MASKS or not r.startswith(("R1g", "R2g")))]

def lin_apply(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    z = ((X.astype(np.float64) - mu) / sd) @ w.T + b                       # locked 3c/4 convention
    return (1 / (1 + np.exp(-z.ravel()))).astype(float)

In [ ]:
def eval_erm_probs(model, df, bs=BATCH):
    model.eval(); ps = []
    for i0 in range(0, len(df), bs):
        chunk = df.iloc[i0:i0 + bs]
        xs = [to_tensor(letterbox_image(read_gray(r["image"]).astype(np.float32)))
              for _, r in chunk.iterrows()]
        x = torch.stack(xs).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                ps.append(torch.sigmoid(model(x)).squeeze(1).float().cpu())
    return torch.cat(ps).numpy().ravel()

import torchvision.models as tvm
erm = tvm.resnet34(weights=None); erm.fc = nn.Linear(erm.fc.in_features, 1)
erm.load_state_dict(load_ckpt(ART["erm_ckpt"])["model"]); erm = erm.to(DEVICE).eval()
for p in erm.parameters():
    p.requires_grad_(False)

## 6. External metrics — thresholds/temperature from TN5000-val only (raw-raw)
AUROC/AUPRC, sens@spec90 at the frozen raw threshold, Brier/ECE on the
val-temperature-calibrated scale (15-bin, pre/post). CIs: 2,000-replicate case
bootstrap. JOINT-img is always descriptive; oracle rows appear iff cohort masks exist.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def fit_temperature(y, p):
    z, yv = logit(np.asarray(p)), np.asarray(y, np.float64)
    grid = np.exp(np.linspace(np.log(0.05), np.log(20.0), 200))
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(yv * np.log(q + 1e-12) + (1 - yv) * np.log(1 - q + 1e-12))
    return float(grid[int(np.argmin([nll(T) for T in grid]))])

def ece(y, p, bins=15):
    y = np.asarray(y, np.float64).ravel(); p = np.asarray(p, np.float64).ravel()
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def brier(y, p):
    return float(np.mean((np.asarray(p).ravel() - np.asarray(y).ravel()) ** 2))

def metric_set(y, p, thr):
    y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
    return {"auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "sens@spec90": float((p[y == 1] >= thr).mean()), "brier": brier(y, p),
            "ece": ece(y, p)}

def boot_ci(y, p, thr, n=2000, seed=SEED):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    rng = np.random.RandomState(seed); N = len(y); acc = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        acc.append([roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx]),
                    (p[idx][y[idx] == 1] >= thr).mean()])
    lo, hi = np.percentile(acc, 2.5, 0), np.percentile(acc, 97.5, 0)
    return lo, hi

def delta_pair_boot(y, pa, pb, n, seed=SEED):
    y = np.asarray(y); rng = np.random.RandomState(seed); N = len(y); d = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2: continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), \
        min(2 * min((d <= 0).mean(), (d >= 0).mean()), 1.0), len(d)

def paired_boot_sens(y, pa, ta, pb, tb, n, seed=SEED):
    y = np.asarray(y); pos = y == 1
    rng = np.random.RandomState(seed)
    sa = (pa[pos] >= ta).astype(float); sb = (pb[pos] >= tb).astype(float)
    M = len(sa); d = []
    for _ in range(n):
        idx = rng.randint(0, M, M)
        d.append(sa[idx].mean() - sb[idx].mean())
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), \
        min(2 * min((d <= 0).mean(), (d >= 0).mean()), 1.0), len(d)

def mcnemar_chi2(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return None, b, c
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return chi2, b, c

In [ ]:
y_ext = THY_BANK["labels"].astype(int)
y_srcv = labels_np("val").astype(int)

P_EXT = {"ERM": eval_erm_probs(erm, manifest5)}
for rep_src in ACTIVE_SRC_REPS:
    P_EXT[MAP_EXT[rep_src]] = lin_apply(rep_src, THY_BANK[MAP_EXT[rep_src]])
P_EXT["JOINT-g"], P_EXT["JOINT-img"] = P_JOINTG_EXT, P_JOINTI_EXT

P_SRCV = {"ERM": np.array(ERMJ["probs"]["val"]).ravel()}
for rep_src in ACTIVE_SRC_REPS:
    P_SRCV[MAP_EXT[rep_src]] = np.array(LINR[rep_src]["probs"]["val"]).ravel()
P_SRCV["JOINT-g"], P_SRCV["JOINT-img"] = (np.array(JP["val"]["JOINTg"]).ravel(),
                                          np.array(JP["val"]["JOINTimg"]).ravel())

# --- joint lineage assertion: recomputed THR/TEMP must match the archived JSON -----
def _lineage(model_key, jp_thr_key):
    thr, _, _ = spec90_threshold(y_srcv, P_SRCV[model_key])
    T = fit_temperature(y_srcv, P_SRCV[model_key])
    assert abs(thr - JP["thr"][jp_thr_key]) <= 1e-6 and abs(T - JP["temp"][jp_thr_key]) <= 1e-6, \
        f"joint lineage failed for {model_key}"
_lineage("JOINT-g", "JOINTg"); _lineage("JOINT-img", "JOINTimg")
print("joint THR/TEMP lineage vs Phase-4 JSON: OK (|delta| <= 1e-6)")

MODEL_ORDER = [m for m in ["ERM", "R0_image", "R1p_lesion", "R2p_lesion_ring",
                           "R1g_gtmask", "R2g_gtmask_ring", "JOINT-g", "JOINT-img"]
               if m in P_EXT]
# --- case-level aggregation (PRIMARY unit; pre-inference amendment 2026-08-22) ----
assert list(manifest5["stem"]) == list(THY_BANK["ids"]), "bank/manifest row order drift"
_case_of_img = manifest5["case"].to_numpy()
_case_order = sorted(case_df5["case"])
_case_pos = {c: i for i, c in enumerate(_case_order)}
y_case = case_df5.set_index("case").loc[_case_order, "label"].astype(int).to_numpy()

def case_mean_scores(p_img):
    acc = np.zeros(len(_case_order)); cnt = np.zeros(len(_case_order))
    for c, v in zip(_case_of_img, np.asarray(p_img, dtype=float)):
        acc[_case_pos[c]] += v; cnt[_case_pos[c]] += 1
    return acc / np.maximum(cnt, 1)

P_CASE = {m: case_mean_scores(P_EXT[m]) for m in MODEL_ORDER}
_first = manifest5["image"].to_numpy() == manifest5.groupby("case")["image"].transform("min").to_numpy()
y_first = y_ext[_first]

THR_EXT, TEMP_EXT, rows = {}, {}, {}
for m in MODEL_ORDER:
    pv_src, pe = P_SRCV[m], P_CASE[m]
    T = fit_temperature(y_srcv, pv_src); TEMP_EXT[m] = T
    pec = 1 / (1 + np.exp(-logit(pe) / T))
    thr, s_v, sp_v = spec90_threshold(y_srcv, pv_src); THR_EXT[m] = thr
    ms = metric_set(y_case, pe, thr)
    ms["brier"] = brier(y_case, pec); ms["ece"] = ece(y_case, pec)
    lo, hi = boot_ci(y_case, pe, thr, n=2000)
    rows[m] = {"model": m, "auroc": round(ms["auroc"], 4),
               "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})", "auprc": round(ms["auprc"], 4),
               "sens@spec90": round(ms["sens@spec90"], 4), "brier": round(ms["brier"], 4),
               "ece_pre": round(ece(y_case, pe), 4), "ece_post": round(ms["ece"], 4),
               "T_srcval": round(T, 3)}
MAIN = pd.DataFrame([rows[m] for m in MODEL_ORDER])
MAIN.to_csv(OUTPUT_ROOT / "phase5_main_table.csv", index=False)
print(f"ThyUS2Path case-level (cases={len(y_case)}, malignant={int(y_case.sum())}; "
      f"images={len(y_ext)}) - case score = mean per-image calibrated prob; THR/TEMP from TN5000 val:")
display(MAIN)
print(f"frozen seg QC: dice {SEG_QC['dice_mean']:.4f} (n/a if no masks; Phase-2 K=20 "
      f"zero-shot refs TNUI {PROTO_REF['TNUI-test']}, DDTI {PROTO_REF['DDTI']}) | "
      f"empty {100*SEG_QC['empty_rate']:.1f}% | area {SEG_QC['area_mean']:.3f}")

# ---- descriptive sensitivity view: first-sorted image per case (original unit) ------
rows_f = []
for m in MODEL_ORDER:
    pf = np.asarray(P_EXT[m], dtype=float)[_first]
    msf = metric_set(y_first, pf, THR_EXT[m])
    rows_f.append({"model": m, "auroc_first_image": round(msf["auroc"], 4),
                   "sens@spec90_first_image": round(msf["sens@spec90"], 4)})
FIRST = pd.DataFrame(rows_f)
FIRST.to_csv(OUTPUT_ROOT / "phase5_firstimage_sensitivity.csv", index=False)
print("sensitivity view (descriptive): first-sorted image per case:")
display(FIRST)

## 7. Family J — pre-declared paired tests (Bonferroni \(\alpha = 0.05/4 = 0.0125\), 5000 resamples)
J1 sens@spec90 R1p − ERM — **PRIMARY** external operating-point replication (+
McNemar, malignant cases); J2 AUROC R1p − ERM (rank); J3 sens@spec90
JOINT-g − R1p (frozen vs joint, H2-analogue); J4 AUROC R1p − R0 (grounding-over-image).
**License**: computed only if `INFERENCE_LICENSE` is true; otherwise the deltas are
computed but reported WITHHELD (descriptive, no significance declared).
All four tests operate on the **case-score unit** (mean of per-image calibrated
probabilities) — the pre-declared primary unit of this cohort.

In [ ]:
tests = [
    ("J1", "sens@spec90", "R1p_lesion", "ERM",      "PRIMARY external operating point (replication)"),
    ("J2", "AUROC",       "R1p_lesion", "ERM",      "rank transfer (secondary)"),
    ("J3", "sens@spec90", "JOINT-g",    "R1p_lesion", "joint vs frozen grounding, external (H2 analogue)"),
    ("J4", "AUROC",       "R1p_lesion", "R0_image", "grounding over whole image, external"),
]
rows = []
for tid, kind, a, b, note in tests:
    if kind == "AUROC":
        m_, lo, hi, p, n = delta_pair_boot(y_case, P_CASE[a], P_CASE[b], N_BOOT_J)
    else:
        m_, lo, hi, p, n = paired_boot_sens(y_case, P_CASE[a], THR_EXT[a], P_CASE[b], THR_EXT[b], N_BOOT_J)
    sig = ("YES" if p < ALPHA_J else "no") if INFERENCE_LICENSE else "WITHHELD"
    rows.append({"test": tid, "metric": kind, "A": a, "B": b, "delta(A-B)": round(m_, 4),
                 "CI95": f"({lo:.4f}..{hi:.4f})", "p_two_sided": round(p, 5),
                 "sig_bonf": sig, "n_boot": n, "note": note})
FAMJ = pd.DataFrame(rows)
FAMJ.to_csv(OUTPUT_ROOT / "phase5_familyJ.csv", index=False)
chi2, b_, c_ = mcnemar_chi2((P_CASE["R1p_lesion"][y_case == 1] >= THR_EXT["R1p_lesion"]),
                            (P_CASE["ERM"][y_case == 1] >= THR_EXT["ERM"]))
print(f"McNemar (J1, positives): R1p right / ERM wrong = {b_}, "
      f"ERM right / R1p wrong = {c_}, chi2 = "
      f"{chi2 if chi2 is None else round(chi2, 2)} (3.84 ~ p<0.05, 6.63 ~ p<0.01)")
print(f"Bonferroni alpha = {ALPHA_J:.4f} | INFERENCE_LICENSE = {INFERENCE_LICENSE}"
      + ("" if INFERENCE_LICENSE else "  -> FAMILY J WITHHELD (descriptive only)"))
display(FAMJ)

## 8. Domain-shift context + run summary
**Send back (one by one):** (1) §1 audit print tail (A1 per-batch coverage, A2 cases, A3
case-exclusions, A4 prevalence, license JSON); (2) polarity-grid verdict (feature: set
POLARITY_VERIFIED then re-run §1c..§7, or accept descriptive); (3) §5 gates
(ERM/R1p integrity + joint lineage); (4) §6 main table + seg-QC lines (frozen +
JOINT-g); (5) §7 family J table + McNemar line; (6) run\_summary.json; (7) wall time.

In [ ]:
src_rows = {"ERM": np.array(ERMJ["probs"]["test"]).ravel()}
for rep_src in ACTIVE_SRC_REPS:
    src_rows[MAP_EXT[rep_src]] = np.array(LINR[rep_src]["probs"]["test"]).ravel()
src_rows["JOINT-g"] = np.array(JP["test"]["JOINTg"]).ravel()
src_rows["JOINT-img"] = np.array(JP["test"]["JOINTimg"]).ravel()
shift = []
for m in MODEL_ORDER:
    a_src = roc_auc_score(y_srct, src_rows[m]); a_ext = roc_auc_score(y_case, P_CASE[m])
    shift.append({"model": m, "AUROC TN5000-test": round(a_src, 4),
                  "AUROC ThyUS2Path": round(a_ext, 4), "shift drop": round(a_src - a_ext, 4)})
SH = pd.DataFrame(shift)
SH.to_csv(OUTPUT_ROOT / "phase5_domain_shift.csv", index=False)
print("domain shift (AUROC, case-level rank metric):")
display(SH)

run = {"phase": "5 - ThyUS2Path case-level external (audit-gated family J)",
       "seed": SEED, "device": str(DEVICE),
       "protocol": {"IMG_SIZE": IMG_SIZE, "PRED_THRESH": PRED_THRESH, "SEG_K": SEG_K,
                    "RING_PX": RING_PX, "FEAT_SCALES": FEAT_SCALES,
                    "N_BOOT_J": N_BOOT_J, "ALPHA_J": ALPHA_J,
                    "PREV_LO": PREV_LO, "PREV_HI": PREV_HI, "MIN_USABLE": MIN_USABLE,
                    "no_refit": True, "joint_lineage_tol": 1e-6},
       "audit": audit_report,
       "gates": {"erm_sanity": float(erm_ref), "r1p_integrity": float(r1p_ref),
                 "joint_lineage_ok": True},
       "cohort": {"unit": "case (mean of per-image calibrated probs)",
                  "n_cases": int(len(y_case)), "malignant_cases": int(y_case.sum()),
                  "prevalence_case": float(y_case.mean()),
                  "n_images": int(len(y_ext)),
                  "per_batch_cases": case_df5.groupby("batch").size().to_dict()},
       "amendments": ["2026-08-22 pre-inference audit amendment: A1b two-table "
                      "per-batch prefix join (batch-namespaced case ids)",
                      "primary unit = case-mean of per-image calibrated probs; "
                      "first-sorted image per case = descriptive sensitivity",
                      "conservative whole-case overlap exclusion (A3)"],
       "seg_qc": {"frozen": SEG_QC, "joint_g": JG_QC, "joint_img": JI_QC},
       "main_table": MAIN.to_dict(orient="records"),
       "firstimage_sensitivity": FIRST.to_dict(orient="records"),
       "familyJ": FAMJ.to_dict(orient="records")}
json.dump(run, open(OUTPUT_ROOT / "run_summary.json", "w"), indent=2,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size/1024:9.1f} KB")